# Comprensión y calidad inicial de los datos

## Objetivo

Construir evidencia reproducible para documentar las 16 variables de los
datasets y evaluar posteriormente su calidad inicial.

Esta fase distingue entre:

- tipo físico inferido por Pandas;
- significado lógico de la variable;
- función analítica;
- unidad o formato;
- disponibilidad por ciudad;
- definiciones verificadas y ambigüedades.

No se limpian, transforman ni concatenan los CSV originales.

## Fuentes

1. Evidencia observada en los seis CSV versionados.
2. Diccionario y supuestos publicados por
   [Inside Airbnb](https://insideairbnb.com/data-assumptions/).
3. Documentación oficial de Airbnb para conceptos de negocio.

Inside Airbnb es una fuente independiente y no está respaldada oficialmente
por Airbnb. Las definiciones se contrastarán con los datos reales.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_directory = project_root / "data" / "raw" / "airbnb"
csv_files = sorted(data_directory.glob("*csv"))

assert len(csv_files) == 6

datasets = {
    csv_file.name: pd.read_csv(
        csv_file,
        low_memory=False,
    )
    for csv_file in csv_files
}

print(f"Datasets cargados: {len(datasets)}")

Datasets cargados: 6


## 1. Identificadores y variables descriptivas

Se examinan `id`, `host_id`, `name` y `host_name` antes de documentar su
significado y función analítica.

In [2]:
first_variable_group = [
    "id",
    "host_id",
    "name",
    "host_name",
]

variable_profile_records = []

for file_name, dataframe in datasets.items():
    for variable_name in first_variable_group:
        column = dataframe[variable_name]

        variable_profile_records.append(
            {
                "file_name": file_name,
                "variable_name": variable_name,
                "pandas_dtype": str(column.dtype),
                "non_null_percentage": round(
                    column.notna().mean() * 100,
                    2,
                ),
                "distinct_values": int(
                    column.nunique(dropna=True)
                ),
                "sample_values": (
                    column.dropna()
                    .astype(str)
                    .drop_duplicates()
                    .head(3)
                    .tolist()
                ),
            }
        )

first_group_profile = pd.DataFrame(
    variable_profile_records
).sort_values(
    ["variable_name", "file_name"],
    ignore_index=True,
)

display(first_group_profile)

,file_name,variable_name,pandas_dtype,non_null_percentage,distinct_values,sample_values
0,NY_airbnb.csv,host_id,int64,100.00,37457,"[2787, 2845, 4632]"
1,london_airbnb.csv,host_id,int64,100.00,53476,"[43039, 54730, 491286]"
2,madrid_airbnb.csv,host_id,int64,100.00,11325,"[13660, 83531, 82175]"
3,milan_airbnb.csv,host_id,int64,100.00,12213,"[13822, 95941, 121663]"
4,sydney_airbnb.csv,host_id,int64,100.00,27219,"[17061, 55948, 59850]"
5,tokyo_airbnb.csv,host_id,int64,100.00,2954,"[151977, 964081, 341577]"
6,NY_airbnb.csv,host_name,str,99.96,11452,"[John, Jennifer, Elisabeth]"
7,london_airbnb.csv,host_name,str,99.99,14573,"[Adriano, Alina, Chil]"
8,madrid_airbnb.csv,host_name,str,97.31,3900,"[Simon, Abdel, Jesus]"
9,milan_airbnb.csv,host_name,str,99.32,2917,"[Francesca, Jeremy, Marta]"


## 2. Variables geográficas

Se examinan `neighbourhood_group`, `neighbourhood`, `latitude` y `longitude`.

El objetivo es distinguir entre etiquetas territoriales y coordenadas, registrar
su cobertura y evitar asumir que los niveles geográficos son equivalentes entre
ciudades.

En esta etapa no se corrigen coordenadas ni se armonizan nombres de barrios.

In [3]:
geographic_variables = [
    "neighbourhood_group",
    "neighbourhood",
    "latitude",
    "longitude",
]

geographic_profile_records = []

for file_name, dataframe in datasets.items():
    for variable_name in geographic_variables:
        if variable_name not in dataframe.columns:
            geographic_profile_records.append(
                {
                    "file_name": file_name,
                    "variable_name": variable_name,
                    "column_exists": False,
                    "pandas_dtype": pd.NA,
                    "non_null_percentage": pd.NA,
                    "distinct_values": pd.NA,
                    "minimum": pd.NA,
                    "maximum": pd.NA,
                    "sample_values": [],
                }
            )
            continue

        column = dataframe[variable_name]
        is_coordinate = variable_name in {
            "latitude",
            "longitude",
        }

        geographic_profile_records.append(
            {
                "file_name": file_name,
                "variable_name": variable_name,
                "column_exists": True,
                "pandas_dtype": str(column.dtype),
                "non_null_percentage": round(
                    column.notna().mean() * 100,
                    2,
                ),
                "distinct_values": int(
                    column.nunique(dropna=True)
                ),
                "minimum": (
                    round(float(column.min()), 6)
                    if is_coordinate
                    else pd.NA
                ),
                "maximum": (
                    round(float(column.max()), 6)
                    if is_coordinate
                    else pd.NA
                ),
                "sample_values": (
                    column.dropna()
                    .astype(str)
                    .drop_duplicates()
                    .head(3)
                    .tolist()
                ),
            }
        )

geographic_profile = pd.DataFrame(
    geographic_profile_records
).sort_values(
    ["variable_name", "file_name"],
    ignore_index=True,
)

display(geographic_profile)

,file_name,variable_name,column_exists,pandas_dtype,non_null_percentage,distinct_values,minimum,maximum,sample_values
0,NY_airbnb.csv,latitude,True,float64,100.0,19048,40.49979,40.91306,"[40.64749, 40.75362, 40.80902]"
1,london_airbnb.csv,latitude,True,float64,100.0,20713,51.29479,51.68169,"[51.46225, 51.56802, 51.51074]"
2,madrid_airbnb.csv,latitude,True,float64,100.0,7536,40.33221,40.56274,"[40.45724, 40.40381, 40.3884]"
3,milan_airbnb.csv,latitude,True,float64,100.0,7419,45.39505,45.53985,"[45.44119, 45.44806, 45.47647]"
4,sydney_airbnb.csv,latitude,True,float64,100.0,36662,-34.135212,-33.389728,"[-33.86515254975741, -33.80092902849084, -33.8..."
5,tokyo_airbnb.csv,latitude,True,float64,100.0,7295,27.07233,35.83243,"[35.67152, 35.717209999999994, 35.742670000000..."
6,NY_airbnb.csv,longitude,True,float64,100.0,14718,-74.24442,-73.71299,"[-73.97237, -73.98377, -73.9419]"
7,london_airbnb.csv,longitude,True,float64,100.0,32079,-0.49668,0.28539,"[-0.11732, -0.11121, -0.19853]"
8,madrid_airbnb.csv,longitude,True,float64,100.0,7595,-3.86391,-3.5319,"[-3.67688, -3.7413, -3.69511]"
9,milan_airbnb.csv,longitude,True,float64,100.0,9134,9.06068,9.27528,"[9.17813, 9.17373, 9.17359]"


### Interpretación de las variables geográficas

`latitude` y `longitude` están informadas en el 100 % de los registros y se
almacenan como números decimales. Se interpretan como coordenadas geográficas,
no como medidas de negocio.

`neighbourhood` está presente y completa en las seis ciudades, pero su cantidad
de categorías varía considerablemente. Los nombres y niveles territoriales no
deben asumirse equivalentes entre ciudades.

`neighbourhood_group` solo contiene información en Nueva York y Madrid. Está
completamente vacía en Londres, Sídney y Tokio, y no existe en Milán.

Sídney presenta una precisión decimal de coordenadas mayor que las demás
ciudades. Tokio contiene coordenadas alejadas de su núcleo urbano habitual.
Ambos patrones se conservarán como observaciones pendientes para la evaluación
de calidad, sin calificarlos todavía como errores.

## 3. Variables de oferta y condiciones

Se examinan `room_type`, `price`, `minimum_nights` y `availability_365`.

Estas variables describen el tipo de alojamiento, el precio publicado, la
estancia mínima y la disponibilidad futura.

Los valores se perfilan sin compararlos directamente entre ciudades, porque
`price` podría estar expresado en monedas locales diferentes.

In [4]:
listing_variables = [
    "room_type",
    "price",
    "minimum_nights",
    "availability_365",
]

numeric_listing_variables = {
    "price",
    "minimum_nights",
    "availability_365",
}

listing_profile_records = []

for file_name, dataframe in datasets.items():
    for variable_name in listing_variables:
        if variable_name not in dataframe.columns:
            listing_profile_records.append(
                {
                    "file_name": file_name,
                    "variable_name": variable_name,
                    "column_exists": False,
                    "pandas_dtype": pd.NA,
                    "non_null_percentage": pd.NA,
                    "distinct_values": pd.NA,
                    "minimum": pd.NA,
                    "median": pd.NA,
                    "maximum": pd.NA,
                    "sample_values": [],
                }
            )
            continue

        column = dataframe[variable_name]
        is_numeric_variable = (
            variable_name in numeric_listing_variables
        )

        if variable_name == "room_type":
            sample_values = sorted(
                column.dropna()
                .astype(str)
                .unique()
                .tolist()
            )
        else:
            sample_values = (
                column.dropna()
                .drop_duplicates()
                .head(5)
                .tolist()
            )

        listing_profile_records.append(
            {
                "file_name": file_name,
                "variable_name": variable_name,
                "column_exists": True,
                "pandas_dtype": str(column.dtype),
                "non_null_percentage": round(
                    column.notna().mean() * 100,
                    2,
                ),
                "distinct_values": int(
                    column.nunique(dropna=True)
                ),
                "minimum": (
                    float(column.min())
                    if is_numeric_variable
                    else pd.NA
                ),
                "median": (
                    float(column.median())
                    if is_numeric_variable
                    else pd.NA
                ),
                "maximum": (
                    float(column.max())
                    if is_numeric_variable
                    else pd.NA
                ),
                "sample_values": sample_values,
            }
        )

listing_profile = pd.DataFrame(
    listing_profile_records
).sort_values(
    ["variable_name", "file_name"],
    ignore_index=True,
)

display(listing_profile)

,file_name,variable_name,column_exists,pandas_dtype,non_null_percentage,distinct_values,minimum,median,maximum,sample_values
0,NY_airbnb.csv,availability_365,True,int64,100.0,366,0.0,45.0,365.0,"[365, 355, 194, 0, 129]"
1,london_airbnb.csv,availability_365,True,int64,100.0,366,0.0,58.0,365.0,"[336, 365, 268, 158, 251]"
2,madrid_airbnb.csv,availability_365,True,int64,100.0,366,0.0,126.0,365.0,"[180, 364, 1, 72, 365]"
3,milan_airbnb.csv,availability_365,True,int64,100.0,366,0.0,123.0,365.0,"[358, 363, 365, 200, 308]"
4,sydney_airbnb.csv,availability_365,True,int64,100.0,366,0.0,36.0,365.0,"[187, 321, 316, 69, 140]"
5,tokyo_airbnb.csv,availability_365,False,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,[]
6,NY_airbnb.csv,minimum_nights,True,int64,100.0,109,1.0,3.0,1250.0,"[1, 3, 10, 45, 2]"
7,london_airbnb.csv,minimum_nights,True,int64,100.0,107,1.0,2.0,1125.0,"[3, 1, 2, 30, 90]"
8,madrid_airbnb.csv,minimum_nights,True,int64,100.0,81,1.0,2.0,1125.0,"[1, 4, 15, 5, 2]"
9,milan_airbnb.csv,minimum_nights,True,int64,100.0,68,1.0,2.0,1124.0,"[4, 1, 2, 3, 20]"
